# 03 - RAG Pipeline: hybrid retrieval + reranking (Rubric item 3 / 25 pts)

**Project:** ShopSense | **Program:** SDAIA Academy - Modern Data Engineering for AI Systems

A support assistant over the ShopSense knowledge base. Every answer must be grounded in
retrieved text and carry citations - and it must refuse to answer when the knowledge base
does not cover the question.

## What this notebook must prove
| Rubric requirement | Where it is proven |
|---|---|
| Document **chunking** | Section 3 - sentence-aware splitter with overlap |
| **Embeddings** | Section 4 - `all-MiniLM-L6-v2` |
| A **real vector store** | Section 5 - ChromaDB persistent collection |
| **Hybrid search** (dense + BM25) | Sections 6.1 / 6.2 |
| Fusion with **Reciprocal Rank Fusion** | Section 6.3 |
| **Reranking** with a cross-encoder | Section 7 - `ms-marco-MiniLM-L-6-v2` |
| Answers **grounded in retrieved context with citations** | Section 8 |
| Retrieval quality measured, not assumed | Section 9 - Hit@3 / MRR across four strategies |

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q "sentence-transformers>=3.0" "chromadb>=1.0,<2.0" rank-bm25

In [ ]:
from pathlib import Path
import json, re, math, datetime, textwrap

PROJECT   = Path('/content/drive/MyDrive/sdaia_capstone')
DOCS_DIR  = PROJECT / 'data' / 'knowledge_base'
REPORTS   = PROJECT / 'reports'
LOCAL_VDB = Path('/content/chroma_store')       # built locally, copied to Drive at the end
DRIVE_VDB = PROJECT / 'vector_store'

for p in [DOCS_DIR, REPORTS, LOCAL_VDB]:
    p.mkdir(parents=True, exist_ok=True)
print('knowledge base ->', DOCS_DIR)

## 2. The knowledge base

Twelve short policy / runbook documents - the kind of material a support agent actually needs.
They are written to disk so the corpus is a versioned project artefact, not a Python literal.

In [ ]:
KB = {}

KB['returns-policy'] = ('Returns and Refunds Policy', '''
ShopSense accepts returns within 14 calendar days of delivery for most products.
The item must be unused, in its original packaging, and accompanied by the original invoice.
Refunds are issued to the original payment method within 7 to 10 business days after the
returned item passes inspection at our Riyadh warehouse.
Electronics may only be returned within 7 days of delivery, and only if the factory seal is intact.
Grocery, beauty and personal-care items cannot be returned once opened, for hygiene reasons.
If the item arrived damaged, the customer must report it within 48 hours of delivery with photos;
in that case ShopSense pays the return shipping and the 14-day window does not apply.
''')

KB['shipping-sla'] = ('Shipping Service Levels', '''
Standard delivery inside Riyadh, Jeddah and Dammam is 1 to 2 business days.
Delivery to Buraydah, Unaizah, Abha and Madinah is 2 to 4 business days.
Remote areas may take up to 6 business days.
Express delivery is available in Riyadh only and guarantees same-day delivery for orders
placed before 13:00 local time, at an extra charge of 25 SAR.
Orders above 200 SAR ship free with standard delivery.
Shipments are handed to the carrier once the order status changes from paid to shipped.
Customers receive a tracking number by SMS at that moment.
''')

KB['payment-failures'] = ('Failed Payments and Declined Cards', '''
A payment can fail for three main reasons: insufficient funds, a card that is not enabled for
online or international transactions, or a failed 3-D Secure verification.
When a payment fails, the order is created but stays in status created and no stock is reserved.
The customer has 30 minutes to retry before the order is cancelled automatically.
If the customer sees a hold on the card for a failed order, that hold is an authorisation and
is released by the issuing bank within 3 to 5 business days. ShopSense never captures it.
Mada, Visa, Mastercard and Apple Pay are supported. Cash on delivery is available for orders
under 1500 SAR inside major cities only.
''')

KB['warranty'] = ('Warranty and Manufacturer Coverage', '''
All electronics sold by ShopSense carry a minimum one-year manufacturer warranty.
Smart watches and wireless earbuds carry a one-year warranty; large home appliances such as
air fryers carry a two-year warranty.
The warranty covers manufacturing defects only. It does not cover water damage, physical damage,
or damage caused by unauthorised repair.
A warranty claim needs the order number and a short description of the fault. The support agent
opens a claim ticket and the customer ships the unit to the service centre at ShopSense expense.
Typical turnaround for a warranty repair is 10 to 15 business days.
''')

KB['order-cancellation'] = ('Cancelling an Order', '''
A customer can cancel an order themselves from the app while the order is in status created or paid.
Once the status becomes shipped, self-service cancellation is disabled and the customer must
refuse delivery or start a return instead.
Cancelling a paid order triggers an automatic refund to the original payment method.
Cash-on-delivery orders that are refused at the door are marked cancelled and no charge is made,
but three refusals within 90 days disable cash on delivery for that customer.
''')

KB['loyalty-program'] = ('ShopSense Rewards', '''
Customers earn 1 reward point for every 10 SAR of delivered order value. Points are credited
7 days after delivery, once the return window for that item has closed.
100 points equal 10 SAR of store credit. Points expire 12 months after they are earned.
The VIP tier starts at 3000 SAR of lifetime value and adds free express delivery in Riyadh
and a 30-day return window instead of 14 days.
Points are not earned on delivery fees, gift cards, or orders paid entirely with store credit.
''')

KB['account-security'] = ('Account Access and Security', '''
Login uses a Saudi mobile number and a one-time password sent by SMS. Passwords are not used.
If the customer changed their number, identity is verified with the national ID on the account
plus the order number of any delivered order from the past six months.
Support agents must never read a one-time password to a customer, and never ask for it.
An account is locked for 30 minutes after five failed one-time password attempts.
Customers can request deletion of their account and personal data from the privacy settings screen;
deletion completes within 30 days, but invoices are retained for 10 years as required by law.
''')

KB['data-dictionary'] = ('Order Data Dictionary', '''
The orders stream carries one event per order state change on the Kafka topic orders.raw.
order_id is the business key and is unique per order. customer_id identifies the buyer.
quantity is a positive integer, unit_price is the price of one unit in the order currency,
and line_total is quantity multiplied by unit_price, computed in the Silver layer.
status is one of created, paid, shipped, delivered or cancelled.
Only paid, shipped and delivered orders count as revenue; created and cancelled orders do not.
order_ts is the moment the order was placed, delivered_ts is the moment it was handed to the customer,
and delivered_ts is never earlier than order_ts.
''')

KB['pipeline-runbook'] = ('Data Pipeline Runbook', '''
The pipeline has five stages: Kafka ingestion, Bronze landing, Silver upsert, Gold aggregation,
and the RAG index refresh.
Records that fail the Pydantic data contract are written to the dead-letter topic orders.dlq
with a rejection_reason field, and to the quarantine folder on disk. They are never dropped silently.
If the quarantine rate for a run exceeds 25 percent, the on-call engineer inspects the producer
before allowing the Silver merge to run.
The Silver layer is maintained with a Delta MERGE keyed on order_id, so replaying the same batch
is safe and does not create duplicates.
Gold tables are rebuilt in full on every run because they are small.
''')

KB['pricing-and-vat'] = ('Pricing, VAT and Invoices', '''
All prices shown on ShopSense include 15 percent value added tax.
The tax invoice is issued when the order reaches status paid and is available as a PDF in the app.
For business customers, a VAT number can be added to the account before checkout so that it appears
on the invoice; it cannot be added afterwards for an already issued invoice.
Prices are quoted in Saudi riyals. Orders placed in USD or AED are converted at the daily rate
captured at the moment of payment and that rate is stored on the order.
''')

KB['delivery-issues'] = ('Missing, Late and Wrong Deliveries', '''
If an order is marked delivered but the customer did not receive it, the agent opens a proof of
delivery request with the carrier. The carrier has 72 hours to respond with a signature or photo.
If the carrier cannot prove delivery, ShopSense reships the order at no cost, or refunds it in full
if the item is out of stock.
An order that is more than 3 business days past its promised delivery date is treated as late; the
customer is entitled to a full refund of any express delivery fee paid.
If the wrong item was delivered, ShopSense arranges pickup and reshipment within 2 business days
and the customer is not charged shipping in either direction.
''')

KB['contact-and-escalation'] = ('Support Hours and Escalation', '''
Support is available Sunday to Thursday from 09:00 to 21:00 and on Saturday from 12:00 to 20:00.
There is no support on Friday. Messages received outside these hours are answered on the next working day.
A ticket is escalated to a senior agent if it is older than 24 hours without a first response, if the
order value is above 2000 SAR, or if the customer has already contacted support three times about the
same order.
Escalated tickets carry a 4-hour response target during working hours.
''')

for doc_id, (title, body) in KB.items():
    (DOCS_DIR / f'{doc_id}.md').write_text(f'# {title}\n\n{body.strip()}\n')

print(f'{len(KB)} documents written to {DOCS_DIR}')
print('total words:', sum(len(b.split()) for _, b in KB.values()))

## 3. Chunking

Fixed-size character splitting cuts sentences in half and hurts retrieval. We split on
sentence boundaries and pack sentences up to a target size, keeping an overlap so a fact that
sits across a boundary is still recoverable.

In [ ]:
CHUNK_CHARS   = 420
OVERLAP_CHARS = 90

SENT_SPLIT = re.compile(r'(?<=[.!?])\s+')

def chunk_document(text, chunk_chars=CHUNK_CHARS, overlap_chars=OVERLAP_CHARS):
    text = re.sub(r'\s+', ' ', text).strip()          # normalise the line wrapping first
    sentences = [s.strip() for s in SENT_SPLIT.split(text) if s.strip()]
    chunks, current = [], ''
    for sent in sentences:
        if current and len(current) + len(sent) + 1 > chunk_chars:
            chunks.append(current.strip())
            tail = current[-overlap_chars:]
            cut = tail.find(' ')
            current = (tail[cut + 1:] if cut != -1 else '') + ' ' + sent
        else:
            current = (current + ' ' + sent).strip()
    if current.strip():
        chunks.append(current.strip())
    return chunks

chunks = []
for doc_id, (title, body) in KB.items():
    for i, text in enumerate(chunk_document(body)):
        chunks.append({
            'chunk_id':  f'{doc_id}::{i:02d}',
            'doc_id':    doc_id,
            'title':     title,
            'position':  i,
            'source':    f'data/knowledge_base/{doc_id}.md',
            'text':      text,
        })

print(f'{len(KB)} documents -> {len(chunks)} chunks')
lens = [len(c['text']) for c in chunks]
print(f'chunk length: min {min(lens)} / mean {sum(lens)//len(lens)} / max {max(lens)} chars')
print('\nexample chunk:')
print(json.dumps(chunks[0], indent=2)[:700])

## 4. Embeddings

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

EMBED_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
embedder = SentenceTransformer(EMBED_MODEL)

texts = [f"{c['title']}. {c['text']}" for c in chunks]     # the title gives the chunk context
embeddings = embedder.encode(texts, batch_size=32, normalize_embeddings=True,
                             show_progress_bar=True)

print('embedding matrix :', embeddings.shape)
print('vector dimension :', embeddings.shape[1])
print('L2 norm of row 0 :', round(float(np.linalg.norm(embeddings[0])), 4), '(normalised)')

## 5. Vector store - ChromaDB (persistent)

In [ ]:
import chromadb

client = chromadb.PersistentClient(path=str(LOCAL_VDB))

COLLECTION = 'shopsense_kb'
try:
    client.delete_collection(COLLECTION)          # idempotent re-runs
except Exception:
    pass

try:
    collection = client.create_collection(name=COLLECTION, metadata={'hnsw:space': 'cosine'})
except Exception:
    collection = client.create_collection(name=COLLECTION)

collection.add(
    ids       = [c['chunk_id'] for c in chunks],
    embeddings= embeddings.tolist(),
    documents = [c['text'] for c in chunks],
    metadatas = [{'doc_id': c['doc_id'], 'title': c['title'],
                  'position': c['position'], 'source': c['source']} for c in chunks],
)

print('collection      :', collection.name)
print('vectors stored  :', collection.count())
print('persisted at    :', LOCAL_VDB)
!ls -la /content/chroma_store

## 6. Hybrid retrieval

Dense search finds paraphrases; BM25 finds exact terms such as `orders.dlq` or `3-D Secure`.
Neither is enough on its own, so we run both and fuse the two ranked lists.

### 6.1 Dense retrieval (vector store)

In [ ]:
by_id = {c['chunk_id']: c for c in chunks}

def dense_search(query, k=10):
    qv = embedder.encode([query], normalize_embeddings=True)[0].tolist()
    res = collection.query(query_embeddings=[qv], n_results=k,
                           include=['documents', 'metadatas', 'distances'])
    out = []
    for cid, dist in zip(res['ids'][0], res['distances'][0]):
        out.append({'chunk_id': cid, 'score': 1.0 - float(dist)})   # cosine distance -> similarity
    return out

for hit in dense_search('how long do I have to send an item back?', k=3):
    c = by_id[hit['chunk_id']]
    print(f"{hit['score']:.3f}  [{c['doc_id']}]  {c['text'][:90]}...")

### 6.2 Keyword retrieval (BM25)

In [ ]:
from rank_bm25 import BM25Okapi

TOKEN = re.compile(r'[a-z0-9\.]+')
def tokenize(t): return TOKEN.findall(t.lower())

corpus_tokens = [tokenize(f"{c['title']} {c['text']}") for c in chunks]
bm25 = BM25Okapi(corpus_tokens)

def bm25_search(query, k=10):
    scores = bm25.get_scores(tokenize(query))
    order = np.argsort(scores)[::-1][:k]
    return [{'chunk_id': chunks[i]['chunk_id'], 'score': float(scores[i])}
            for i in order if scores[i] > 0]

for hit in bm25_search('orders.dlq rejection_reason', k=3):
    c = by_id[hit['chunk_id']]
    print(f"{hit['score']:.3f}  [{c['doc_id']}]  {c['text'][:90]}...")

### 6.3 Reciprocal Rank Fusion

RRF combines the two lists by **rank**, not by score, so we never have to normalise a cosine
similarity against a BM25 score:

$$\text{RRF}(d)=\sum_{r \in \text{rankers}} \frac{1}{k + \text{rank}_r(d)} \qquad k=60$$

In [ ]:
RRF_K = 60

def reciprocal_rank_fusion(ranked_lists, k=RRF_K, top_n=10):
    fused = {}
    for lst in ranked_lists:
        for rank, hit in enumerate(lst, start=1):
            cid = hit['chunk_id']
            fused[cid] = fused.get(cid, 0.0) + 1.0 / (k + rank)
    ordered = sorted(fused.items(), key=lambda kv: kv[1], reverse=True)[:top_n]
    return [{'chunk_id': cid, 'score': s} for cid, s in ordered]

def hybrid_search(query, k_each=10, top_n=10):
    return reciprocal_rank_fusion([dense_search(query, k_each), bm25_search(query, k_each)],
                                  top_n=top_n)

q = 'what happens to a record that fails validation?'
print('DENSE  :', [by_id[h['chunk_id']]['doc_id'] for h in dense_search(q, 5)])
print('BM25   :', [by_id[h['chunk_id']]['doc_id'] for h in bm25_search(q, 5)])
print('HYBRID :', [by_id[h['chunk_id']]['doc_id'] for h in hybrid_search(q, top_n=5)])

## 7. Reranking with a cross-encoder

The retrievers score the query and the document separately. A cross-encoder reads the pair
**together**, which is far more accurate - but too slow to run over the whole corpus.
So it only reorders the top candidates the hybrid stage produced.

In [ ]:
from sentence_transformers import CrossEncoder

RERANK_MODEL = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
reranker = CrossEncoder(RERANK_MODEL)

def sigmoid(x): return 1.0 / (1.0 + math.exp(-x))

def rerank(query, candidates, top_n=4):
    pairs = [[query, by_id[c['chunk_id']]['text']] for c in candidates]
    logits = reranker.predict(pairs)
    scored = [{'chunk_id': c['chunk_id'],
               'rerank_score': float(s),
               'confidence': sigmoid(float(s))}
              for c, s in zip(candidates, logits)]
    return sorted(scored, key=lambda d: d['rerank_score'], reverse=True)[:top_n]

def retrieve(query, candidates_n=10, top_n=4):
    return rerank(query, hybrid_search(query, top_n=candidates_n), top_n=top_n)

q = 'can I return a phone after ten days?'
print('before rerank :', [by_id[h['chunk_id']]['doc_id'] for h in hybrid_search(q, top_n=6)])
print('after  rerank :')
for h in retrieve(q):
    c = by_id[h['chunk_id']]
    print(f"  conf {h['confidence']:.3f}  [{c['doc_id']}]  {c['text'][:80]}...")

## 8. Grounded answering with citations

Rules the assistant follows:

1. Answer **only** from the retrieved chunks.
2. Every sentence carries a citation marker `[S1]`, `[S2]`, ... pointing at the chunk it came from.
3. If the best reranked chunk is below the confidence floor, **refuse** instead of guessing.

In [ ]:
CONFIDENCE_FLOOR = 0.30

def build_context(hits):
    blocks = []
    for i, h in enumerate(hits, start=1):
        c = by_id[h['chunk_id']]
        blocks.append({'tag': f'S{i}', 'title': c['title'], 'source': c['source'],
                       'chunk_id': c['chunk_id'], 'text': c['text'],
                       'confidence': h['confidence']})
    return blocks

def extractive_answer(query, blocks, max_sentences=4):
    # score every sentence of the retrieved context against the query, keep the best few,
    # and keep each sentence attached to the block it came from so the citation is honest.
    cand = []
    for b in blocks:
        for sent in [s.strip() for s in SENT_SPLIT.split(b['text']) if len(s.strip()) > 30]:
            cand.append((sent, b['tag'], b['confidence']))
    if not cand:
        return None
    qv = embedder.encode([query], normalize_embeddings=True)[0]
    sv = embedder.encode([c[0] for c in cand], normalize_embeddings=True)
    sims = sv @ qv
    order = np.argsort(sims)[::-1][:max_sentences]
    order = sorted(order, key=lambda i: (cand[i][1], i))       # keep source order readable
    return ' '.join(f'{cand[i][0]} [{cand[i][1]}]' for i in order)

def ask(query, top_n=4, verbose=True):
    hits = retrieve(query, top_n=top_n)
    blocks = build_context(hits)
    best = blocks[0]['confidence'] if blocks else 0.0

    if best < CONFIDENCE_FLOOR:
        answer = ('I could not find this in the ShopSense knowledge base, so I will not guess. '
                  f'(best retrieval confidence {best:.2f} < floor {CONFIDENCE_FLOOR})')
        grounded = False
    else:
        answer = extractive_answer(query, blocks)
        grounded = True

    result = {'question': query, 'answer': answer, 'grounded': grounded,
              'top_confidence': round(best, 3),
              'citations': [{'tag': b['tag'], 'title': b['title'], 'source': b['source'],
                             'chunk_id': b['chunk_id'], 'confidence': round(b['confidence'], 3)}
                            for b in blocks]}
    if verbose:
        print('Q:', query)
        print('\nA:', textwrap.fill(answer, 100))
        if grounded:
            print('\nSources:')
            for b in blocks:
                print(f"  [{b['tag']}] {b['title']}  ({b['source']}, chunk {b['chunk_id']}, "
                      f"confidence {b['confidence']:.2f})")
        print('=' * 100)
    return result

_ = ask('How many days do I have to return a pair of earbuds?')

In [ ]:
demo_questions = [
    'My card was declined but the bank put a hold on my money. What happens now?',
    'How long does delivery to Buraydah take?',
    'What happens to a record that fails the data contract?',
    'How many reward points do I get and when do they expire?',
    'Is support available on Friday?',
    'What is the capital of Japan?',        # out of scope - must be refused
]
answers = [ask(q) for q in demo_questions]

### 8.1 Optional - generative answer with an LLM

In [ ]:
# Fully optional. The pipeline above already produces grounded, cited answers without any API key.
# If you have a Gemini key, put it in Colab Secrets (key icon on the left) under the name GEMINI_API_KEY.
def llm_answer(query, top_n=4):
    try:
        from google.colab import userdata
        import google.generativeai as genai
        genai.configure(api_key=userdata.get('GEMINI_API_KEY'))
    except Exception as exc:
        print('No LLM configured, skipping generative answer:', type(exc).__name__)
        return None

    blocks = build_context(retrieve(query, top_n=top_n))
    context = '\n\n'.join(f"[{b['tag']}] ({b['title']}) {b['text']}" for b in blocks)
    prompt = (
        'Answer the question using ONLY the context below. '
        'Cite the tag [S1], [S2] ... after every claim. '
        "If the context does not contain the answer, reply exactly: NOT IN CONTEXT.\n\n"
        f'CONTEXT:\n{context}\n\nQUESTION: {query}\nANSWER:'
    )
    model = genai.GenerativeModel('gemini-1.5-flash')
    text = model.generate_content(prompt).text
    print('Q:', query, '\n\nA:', text)
    return text

_ = llm_answer('Can I cancel an order that has already shipped?')

## 9. Does the hybrid + rerank stack actually help?

A small labelled set: each question is tagged with the document that *should* be retrieved.
We compare four configurations on **Hit@3** (is the right document in the top 3?) and
**MRR** (how high up is it?).

In [ ]:
EVAL = [
    ('Can I return opened shampoo?',                                'returns-policy'),
    ('How fast is express delivery in Riyadh?',                     'shipping-sla'),
    ('Why is there a hold on my card after a failed payment?',      'payment-failures'),
    ('How long is the warranty on an air fryer?',                   'warranty'),
    ('Where do rejected records go?',                               'pipeline-runbook'),
    ('Does the displayed price include VAT?',                       'pricing-and-vat'),
    ('When are reward points credited?',                            'loyalty-program'),
    ('The courier says delivered but I have nothing.',              'delivery-issues'),
    ('What counts as revenue in the orders data?',                  'data-dictionary'),
    ('When does a ticket get escalated?',                           'contact-and-escalation'),
]

def docs_of(hits):
    seen, out = set(), []
    for h in hits:
        d = by_id[h['chunk_id']]['doc_id']
        if d not in seen:
            seen.add(d); out.append(d)
    return out

STRATEGIES = {
    'dense only'      : lambda q: docs_of(dense_search(q, 10)),
    'bm25 only'       : lambda q: docs_of(bm25_search(q, 10)),
    'hybrid (RRF)'    : lambda q: docs_of(hybrid_search(q, top_n=10)),
    'hybrid + rerank' : lambda q: docs_of(rerank(q, hybrid_search(q, top_n=10), top_n=10)),
}

rows = []
for name, fn in STRATEGIES.items():
    hits3, rr = 0, 0.0
    for question, gold in EVAL:
        ranked = fn(question)
        if gold in ranked[:3]:
            hits3 += 1
        if gold in ranked:
            rr += 1.0 / (ranked.index(gold) + 1)
    rows.append({'strategy': name,
                 'Hit@3': round(hits3 / len(EVAL), 3),
                 'MRR':   round(rr / len(EVAL), 3)})

import pandas as pd
results = pd.DataFrame(rows)
display(results)

In [ ]:
best = results.sort_values('MRR', ascending=False).iloc[0]
print(f"best configuration: {best['strategy']}  (Hit@3 {best['Hit@3']}, MRR {best['MRR']})")
print(f"lift in MRR over dense-only: "
      f"{best['MRR'] - results.loc[results.strategy == 'dense only', 'MRR'].iloc[0]:+.3f}")

## 10. Persist the vector store and the stage report

In [ ]:
import shutil
if DRIVE_VDB.exists():
    shutil.rmtree(DRIVE_VDB)
shutil.copytree(LOCAL_VDB, DRIVE_VDB)
print('vector store copied to', DRIVE_VDB)

run_id = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
report = {
    'run_id': run_id,
    'stage': 'rag',
    'documents': len(KB),
    'chunks': len(chunks),
    'chunking': {'strategy': 'sentence-aware packing',
                 'target_chars': CHUNK_CHARS, 'overlap_chars': OVERLAP_CHARS},
    'embedding_model': EMBED_MODEL,
    'embedding_dim': int(embeddings.shape[1]),
    'vector_store': {'engine': 'chromadb', 'collection': COLLECTION,
                     'vectors': collection.count(), 'path': str(DRIVE_VDB)},
    'retrieval': {'dense': 'cosine kNN', 'sparse': 'BM25Okapi',
                  'fusion': f'reciprocal rank fusion (k={RRF_K})',
                  'reranker': RERANK_MODEL, 'confidence_floor': CONFIDENCE_FLOOR},
    'evaluation': rows,
    'demo_answers': answers,
    'finished_at': datetime.datetime.now(datetime.timezone.utc).isoformat(),
}
out = REPORTS / f'rag_report_{run_id}.json'
out.write_text(json.dumps(report, indent=2, default=str))
print('saved ->', out)
print(json.dumps({k: v for k, v in report.items() if k != 'demo_answers'}, indent=2, default=str))

## What is done, and what is left

Done: **Ingestion (20)**, **Delta Lakehouse (25)**, **RAG (25)** = 70 of 100 - already above the pass mark.

Tomorrow: **Orchestration - Airflow DAG (15)** and **Quality Gate + Lineage - Great Expectations
and OpenLineage (15)**. Those two wrap the three notebooks above into one runnable pipeline where
a failed expectation stops the run before Gold or the RAG refresh.